# Transformer / Swin-T-FPN

Patch embedding, window and shifted-window stages, FPN outputs, and optional attention rollout. Strong context modeling; higher memory and dependency complexity.

License and exact weight provenance are recorded in `LICENSES.md` and each run manifest. Approximate GPU requirements depend strongly on resolution, batch size, AMP, and the active Colab GPU.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

SMOKE_TEST = os.environ.get("SMOKE_TEST", "0").lower() in {"1", "true", "yes", "on"}
try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
REPOSITORY_URL = os.environ.get(
    "BENCHMARK_REPOSITORY_URL",
    "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git",
)
REPOSITORY_BRANCH = os.environ.get("BENCHMARK_REPOSITORY_BRANCH", "main")
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/aerial-object-detection-benchmark")
    if not (REPO_DIR / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(REPO_DIR)],
            check=True,
        )
else:
    REPO_DIR = Path(os.environ.get("BENCHMARK_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_DIR / "pyproject.toml").is_file():
    raise RuntimeError(f"Repository root is invalid: {REPO_DIR}")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if IS_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-dataset-colab.txt"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
DRIVE_ROOT = os.environ.get(
    "VISDRONE_DRIVE_ROOT",
    "/content/drive/MyDrive/visdrone_architecture_benchmark"
    if IS_COLAB
    else str(REPO_DIR / ".notebook-smoke"),
)
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
print({"repo": str(REPO_DIR), "storage": str(paths.root), "smoke_test": SMOKE_TEST})
collect_environment()
if not SMOKE_TEST:
    MMDET_ROOT = Path(os.environ.get("MMDET_ROOT", "/content/mmdetection"))
    if not (MMDET_ROOT / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "v3.3.0",
             "https://github.com/open-mmlab/mmdetection.git", str(MMDET_ROOT)],
            check=True,
        )
    os.environ["MMDET_ROOT"] = str(MMDET_ROOT)


In [ ]:
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
collect_environment()

## Editable experiment configuration

In [ ]:
MODEL_ID = "faster_rcnn_swin_t"
DATASET_TRACK = "2class"
IMAGE_SIZE = 1024
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
NUM_EPOCHS = 100
SEED = 42
USE_AMP = True
RESUME_RUN_ID = None
print(dict(MODEL_ID=MODEL_ID, DATASET_TRACK=DATASET_TRACK, IMAGE_SIZE=IMAGE_SIZE, EFFECTIVE_BATCH_SIZE=EFFECTIVE_BATCH_SIZE))
if SMOKE_TEST:
    NUM_EPOCHS = 1
    BATCH_SIZE = 1
print('SMOKE_TEST stops before model construction/training:', SMOKE_TEST)

if SMOKE_TEST:
    NUM_EPOCHS = 1
    BATCH_SIZE = 1
print('SMOKE_TEST stops before model construction/training:', SMOKE_TEST)

if SMOKE_TEST:
    NUM_EPOCHS = 1
    BATCH_SIZE = 1
print('SMOKE_TEST stops before model construction/training:', SMOKE_TEST)

if SMOKE_TEST:
    NUM_EPOCHS = 1
    BATCH_SIZE = 1
print('SMOKE_TEST stops before model construction/training:', SMOKE_TEST)

if SMOKE_TEST:
    NUM_EPOCHS = 1
    BATCH_SIZE = 1
print('SMOKE_TEST stops before model construction/training:', SMOKE_TEST)

if SMOKE_TEST:
    NUM_EPOCHS = 1
    BATCH_SIZE = 1
print('SMOKE_TEST stops before model construction/training:', SMOKE_TEST)

if SMOKE_TEST:
    NUM_EPOCHS = 1
    BATCH_SIZE = 1
print('SMOKE_TEST stops before model construction/training:', SMOKE_TEST)

if SMOKE_TEST:
    NUM_EPOCHS = 1
    BATCH_SIZE = 1
print('SMOKE_TEST stops before model construction/training:', SMOKE_TEST)

if SMOKE_TEST:
    NUM_EPOCHS = 1
    BATCH_SIZE = 1
print('SMOKE_TEST stops before model construction/training:', SMOKE_TEST)


## Dataset validation

Validation checks image existence, dimensions, category IDs, bbox coordinates, zero-area boxes, and class coverage. Statistics expose class counts, size distributions, and objects per image.

In [ ]:
from src.notebook_utils import preflight_dataset
report = preflight_dataset(paths, DATASET_TRACK, minimum_free_gb=0 if SMOKE_TEST else 5)
print(report)
report.raise_for_errors()


## Model construction and introspection

The training command saves architecture, parameter totals, trainable/frozen totals, runtime config, and environment. After a first checkpoint, use notebook 08 for feature shapes, stage strides, FLOPs/MACs, and actual module names.

## Baseline training

Every epoch logs loss, LR, duration, gradient norm, memory, and validation metrics supported by the integration. `last.pth`, `best_map.pth`, and `best_aptiny.pth` are saved under the standardized run directory. Pass `RESUME_RUN_ID` after a Colab disconnect.

In [ ]:
import shlex
cmd = f"python scripts/train.py --drive-root '{DRIVE_ROOT}' --model-id {MODEL_ID} --dataset-track {DATASET_TRACK} --image-size {IMAGE_SIZE} --batch-size {BATCH_SIZE} --gradient-accumulation-steps {GRADIENT_ACCUMULATION_STEPS} --epochs {NUM_EPOCHS} --seed {SEED}"
if not USE_AMP:
    cmd += " --no-amp"
if RESUME_RUN_ID:
    cmd += f" --resume-run-id {RESUME_RUN_ID}"
print(cmd)
if SMOKE_TEST:
    print("SMOKE_TEST: command validated; expensive training not started.")
else:
    from src.notebook_utils import require_gpu, require_model_environment
    require_model_environment("rtdetr" if MODEL_ID == "rtdetrv2_l" else "openmmlab")
    require_gpu(MODEL_ID)
    subprocess.run(shlex.split(cmd), check=True)


## Swin/FPN visualization

Capture patch embedding, hierarchical stages, FPN, RPN, and RoI modules. Shifted-window internals are selected only from real installed module names.


In [ ]:
from src.training.checkpointing import RunRegistry
from src.models.registry import create_adapter
from src.utils.serialization import read_yaml
from src.evaluation.visualization import select_module_names, capture_module_outputs, plot_activation_views, draw_predictions
from src.data.dataloaders import CocoDetectionRecords
from IPython.display import display
registry = RunRegistry(paths)
completed = registry.list_available_runs(MODEL_ID, DATASET_TRACK)
if completed:
    run = completed[0]
    run_dir = paths.run_dir(MODEL_ID, run["run_id"])
    model_cfg = read_yaml(run_dir / "model_config.yaml")
    model_cfg["input_resolution"] = run["input_resolution"]
    if run["framework"] in {"mmdetection", "vmamba_mmdetection"}:
        model_cfg["resolved_framework_config"] = str(run_dir / "runtime_config.py")
    adapter = create_adapter(MODEL_ID)
    model = adapter.load_model(registry.load_checkpoint_from_registry(run["run_id"]), model_cfg)
    names = select_module_names(model, ['patch_embed', 'backbone.stages', 'neck', 'rpn_head', 'roi_head'], limit=18)
    print("Hooked modules:", *names, sep="\n- ")
    records = CocoDetectionRecords(paths.coco(DATASET_TRACK)/"val", paths.coco(DATASET_TRACK)/"annotations/instances_val.json")
    sample = records[0]["image"]
    outputs, handles = capture_module_outputs(model, names)
    prediction = adapter.predict([sample])[0]
    for handle in handles: handle.remove()
    display(draw_predictions(sample, prediction, run["class_names"], threshold=0.25))
    try:
        display(plot_activation_views(outputs))
    except RuntimeError as error:
        print(error)
else:
    print("Train or resume a completed run before executing this visualization cell.")


## Learning-rate selection

Use the shared `12_learning_rate_search.ipynb`, then `13_full_dataset_finetune.ipynb`. The legacy multidimensional Optuna search has been removed.


In [ ]:
print('LR search: notebooks/12_learning_rate_search.ipynb')
print('Final training: notebooks/13_full_dataset_finetune.ipynb')


## Final summary

The printed manifest contains best epoch, mAP, APtiny, total time, parameter counts, checkpoint paths, framework versions, GPU, and registered run ID.

In [ ]:
from src.training.checkpointing import RunRegistry
runs = RunRegistry(paths).list_available_runs(MODEL_ID, DATASET_TRACK, status=None)
runs[:3]